# SMILES Generation from InChI

The purpose of this notebook is to generate a processed version of the dataset that will have a SMILES column

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%load_ext dotenv
%dotenv

In [ ]:
import sys
import pandas as pd
import numpy as np
from pyprojroot import here
from dl_hplc_smrt.utils import inchi_to_smiles

## Path and Files Names

In [4]:
RAW_DATA_PATH = here() / "data" / "raw"
PROCESSED_DATA_PATH = here() / "data" / "processed"

In [5]:
ORIGINAL_SMRT_FILE = RAW_DATA_PATH / "SMRT_dataset.csv"
PROCESSED_SMRT_FILE = PROCESSED_DATA_PATH / "proc_SMRT_dataset.csv"

## Reading the original dataset

In [6]:
smrt_dataset_df = pd.read_csv(ORIGINAL_SMRT_FILE, sep=";")
print(smrt_dataset_df.shape)
smrt_dataset_df.head()

(80038, 3)


,pubchem,rt,inchi
0,5139,93.5,"InChI=1S/C3H8N2S/c1-2-6-3(4)5/h2H2,1H3,(H3,4,5)"
1,3505,687.8,InChI=1S/C19H25Cl2N3O3/c1-27-19(26)23-8-9-24(1...
2,2159,590.7,InChI=1S/C17H27N3O4S/c1-4-20-8-6-7-12(20)11-19...
3,1340,583.6,InChI=1S/C9H7NO2/c11-8-3-1-2-7-6(8)4-5-10-9(7)...
4,3344,579.0,InChI=1S/C15H20N2O2/c18-14-16-12-15(19-14)7-10...


Apparently there are a few repeated InChI strings:

In [7]:
smrt_dataset_df.describe(include=['category', 'object'])

/tmp/ipykernel_49004/2913326157.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  smrt_dataset_df.describe(include=['category', 'object'])


,inchi
count,80038
unique,80019
top,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...
freq,3


Let's investigate that

In [8]:
inchi_counts = smrt_dataset_df['inchi'].value_counts()
repeated_inchis = inchi_counts[inchi_counts > 1]

In [9]:
repeated_inchis = repeated_inchis.index
repeated_inchis

Index(['InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-13)21-17(20)16(11-19)12-5-3-2-4-6-12/h2-6,13-16,19H,7-11H2,1H3/t13-,14+,15-,16-/m1/s1',
       'InChI=1S/C15H20O3/c1-9-5-4-8-15(3)13(18-15)12-11(7-6-9)10(2)14(16)17-12/h5,11-13H,2,4,6-8H2,1,3H3/b9-5-/t11-,12-,13+,15+/m0/s1',
       'InChI=1S/C4H8N2O2/c5-3-1-2-6(8)4(3)7/h3,8H,1-2,5H2/t3-/m0/s1',
       'InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-10-15(11-13-18)14-6-2-1-3-7-14/h1-3,6-7,15-17,19H,4-5,8-13H2/t16-,17+/m0/s1',
       'InChI=1S/C26H28ClNO/c1-3-28(4-2)19-20-29-24-17-15-22(16-18-24)25(21-11-7-5-8-12-21)26(27)23-13-9-6-10-14-23/h5-18H,3-4,19-20H2,1-2H3/b26-25-',
       'InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15(13)22-16)21-17(20)12(9-19)10-5-3-2-4-6-10/h2-6,11-16,19H,7-9H2,1H3/t11-,12-,13+,14-,15+,16-/m1/s1',
       'InChI=1S/C13H15NO2/c15-12-7-9-4-5-10-8-13(9,16-12)11-3-1-2-6-14(10)11/h4-5,7,10-11H,1-3,6,8H2/t10-,11-,13+/m1/s1',
       'InChI=1S/C29H41F2N5O/c1-19(2)27-34-33-20(3)36(27)25-17-23-9-10-24(18-25)35(23)16-1

In [10]:
smrt_dataset_repeated_df = smrt_dataset_df.loc[smrt_dataset_df["inchi"].isin(repeated_inchis), :]
smrt_dataset_repeated_df.head()

,pubchem,rt,inchi
11,1232,620.6,"InChI=1S/C4H8N2O2/c5-3-1-2-6(8)4(3)7/h3,8H,1-2..."
42,3661,638.0,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...
54,5662,679.1,InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-1...
176,2800,1008.2,InChI=1S/C26H28ClNO/c1-3-28(4-2)19-20-29-24-17...
615,2723877,581.3,InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15...


In [11]:
smrt_dataset_repeated_aggregated_df = smrt_dataset_repeated_df.loc[:,['inchi','rt']].groupby('inchi').agg(
    rt_min = ('rt', 'min'),
    rt_max = ('rt', 'max'),
    rt_mean = ('rt', 'mean')
    ).reset_index()

In [12]:
smrt_dataset_repeated_aggregated_df['pubchem'] = smrt_dataset_repeated_aggregated_df['inchi'].apply(lambda inchi: smrt_dataset_repeated_df.loc[smrt_dataset_repeated_df["inchi"]==inchi,"pubchem"].to_list())

In [13]:
smrt_dataset_repeated_aggregated_df.head(10)

,inchi,rt_min,rt_max,rt_mean,pubchem
0,InChI=1S/C13H13N3/c1-2-16-13-5-11-9-3-8(6-14-7...,92.3,101.1,96.700000,"[5310966, 170361]"
1,"InChI=1S/C13H15NO2/c15-12-7-9-4-5-10-8-13(9,16...",86.4,94.5,90.450000,"[267769, 6338099]"
2,InChI=1S/C15H20O3/c1-9-5-4-8-15(3)13(18-15)12-...,843.6,972.3,904.766667,"[5420804, 7251185, 927704]"
3,InChI=1S/C16H24O4/c1-11-5-3-2-4-6-12-9-13(17)1...,732.2,781.6,756.900000,"[5287620, 6436187]"
4,InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15...,581.3,619.4,600.350000,"[2723877, 3000322]"
5,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...,613.3,667.1,639.466667,"[3661, 64692, 174174]"
6,InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-1...,679.1,701.9,690.500000,"[5662, 16411702]"
7,InChI=1S/C21H24N2O2/c1-3-14-10-13-11-21(20(24)...,681.1,727.8,704.450000,"[5458190, 72315]"
8,InChI=1S/C21H27NO4/c23-14-5-4-13-10-16-21(25)7...,618.9,635.1,627.000000,"[11957646, 16741281]"
9,InChI=1S/C23H27N5/c1-19-23(20(2)28(25-19)22-11...,687.2,739.4,713.300000,"[6878030, 1356208]"


Most of the values for the different molecules with shared InChI are rather similar. Furthermore, visual inspection of a few of those molecules hint towrads them being different only in terms of chiral centers (sometimes they may correspond to different enantiomers, sometimes some of the centers appear to be racemic). We will not deal with stereochemistry, so we will input the mean instead and keep just one copy.

## Remove Duplicates and Input the Mean Retention Time

Remove duplicates keeping just the first instance

In [14]:
print(smrt_dataset_df.shape)
smrt_dataset_df.drop_duplicates(subset=['inchi'], inplace=True)
print(smrt_dataset_df.shape)

(80038, 3)
(80019, 3)


Input the mean for those that had duplicates

In [15]:
mask_duplicated = smrt_dataset_df["inchi"].isin(smrt_dataset_repeated_aggregated_df["inchi"])

In [16]:
smrt_dataset_df.loc[mask_duplicated, :].head(10)

,pubchem,rt,inchi
11,1232,620.6,"InChI=1S/C4H8N2O2/c5-3-1-2-6(8)4(3)7/h3,8H,1-2..."
42,3661,638.0,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...
54,5662,679.1,InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-1...
176,2800,1008.2,InChI=1S/C26H28ClNO/c1-3-28(4-2)19-20-29-24-17...
615,2723877,581.3,InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15...
708,267769,86.4,"InChI=1S/C13H15NO2/c15-12-7-9-4-5-10-8-13(9,16..."
1489,3002977,665.8,InChI=1S/C29H41F2N5O/c1-19(2)27-34-33-20(3)36(...
3896,5287620,781.6,InChI=1S/C16H24O4/c1-11-5-3-2-4-6-12-9-13(17)1...
3934,5310966,101.1,InChI=1S/C13H13N3/c1-2-16-13-5-11-9-3-8(6-14-7...
4147,5420804,972.3,InChI=1S/C15H20O3/c1-9-5-4-8-15(3)13(18-15)12-...


Let's copy the rt values to old_rt

In [17]:
smrt_dataset_df["old_rt"] = smrt_dataset_df["rt"]

In [18]:
smrt_dataset_df.loc[mask_duplicated, ["inchi","rt", "old_rt"]]

,inchi,rt,old_rt
11,"InChI=1S/C4H8N2O2/c5-3-1-2-6(8)4(3)7/h3,8H,1-2...",620.6,620.6
42,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...,638.0,638.0
54,InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-1...,679.1,679.1
176,InChI=1S/C26H28ClNO/c1-3-28(4-2)19-20-29-24-17...,1008.2,1008.2
615,InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15...,581.3,581.3
708,"InChI=1S/C13H15NO2/c15-12-7-9-4-5-10-8-13(9,16...",86.4,86.4
1489,InChI=1S/C29H41F2N5O/c1-19(2)27-34-33-20(3)36(...,665.8,665.8
3896,InChI=1S/C16H24O4/c1-11-5-3-2-4-6-12-9-13(17)1...,781.6,781.6
3934,InChI=1S/C13H13N3/c1-2-16-13-5-11-9-3-8(6-14-7...,101.1,101.1
4147,InChI=1S/C15H20O3/c1-9-5-4-8-15(3)13(18-15)12-...,972.3,972.3


Let's map the mean rt values for those rows whose inchi is in the duplicated list

In [19]:
rt_map = smrt_dataset_repeated_aggregated_df.set_index("inchi")["rt_mean"]
smrt_dataset_df["rt"] = (
    smrt_dataset_df["inchi"].map(rt_map).fillna(smrt_dataset_df["old_rt"])
)

In [20]:
smrt_dataset_df.loc[mask_duplicated, ["inchi","rt", "old_rt"]]

,inchi,rt,old_rt
11,"InChI=1S/C4H8N2O2/c5-3-1-2-6(8)4(3)7/h3,8H,1-2...",623.700000,620.6
42,InChI=1S/C17H23NO3/c1-18-13-7-8-14(18)10-15(9-...,639.466667,638.0
54,InChI=1S/C17H25NO/c19-17-9-5-4-8-16(17)18-12-1...,690.500000,679.1
176,InChI=1S/C26H28ClNO/c1-3-28(4-2)19-20-29-24-17...,1001.900000,1008.2
615,InChI=1S/C17H21NO4/c1-18-13-7-11(8-14(18)16-15...,600.350000,581.3
708,"InChI=1S/C13H15NO2/c15-12-7-9-4-5-10-8-13(9,16...",90.450000,86.4
1489,InChI=1S/C29H41F2N5O/c1-19(2)27-34-33-20(3)36(...,670.150000,665.8
3896,InChI=1S/C16H24O4/c1-11-5-3-2-4-6-12-9-13(17)1...,756.900000,781.6
3934,InChI=1S/C13H13N3/c1-2-16-13-5-11-9-3-8(6-14-7...,96.700000,101.1
4147,InChI=1S/C15H20O3/c1-9-5-4-8-15(3)13(18-15)12-...,904.766667,972.3


Drop the old rt column

In [21]:
smrt_dataset_df.drop(["old_rt"], inplace=True, axis=1)
smrt_dataset_df.head()

,pubchem,rt,inchi
0,5139,93.5,"InChI=1S/C3H8N2S/c1-2-6-3(4)5/h2H2,1H3,(H3,4,5)"
1,3505,687.8,InChI=1S/C19H25Cl2N3O3/c1-27-19(26)23-8-9-24(1...
2,2159,590.7,InChI=1S/C17H27N3O4S/c1-4-20-8-6-7-12(20)11-19...
3,1340,583.6,InChI=1S/C9H7NO2/c11-8-3-1-2-7-6(8)4-5-10-9(7)...
4,3344,579.0,InChI=1S/C15H20N2O2/c18-14-16-12-15(19-14)7-10...


## Get SMILES from InChI

In [22]:
smrt_dataset_df['SMILES'] = smrt_dataset_df["inchi"].apply(inchi_to_smiles)

[18:37:10] Explicit valence for atom # 14 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 14 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 16 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 16 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 18 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 18 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 17 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 17 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 15 N, 4, is greater than permitted
[18:37:10] ERROR: Explicit valence for atom # 15 N, 4, is greater than permitted

[18:37:10] Explicit valence for atom # 16 N, 4, is greater than 

As it was evident by the logs, not all molecules could be converted to SMILES

In [23]:
smrt_dataset_df.describe(include=['category', 'object'])

/tmp/ipykernel_49004/2913326157.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  smrt_dataset_df.describe(include=['category', 'object'])


,inchi,SMILES
count,80019,79938
unique,80019,79938
top,"InChI=1S/C3H8N2S/c1-2-6-3(4)5/h2H2,1H3,(H3,4,5)",CCSC(=N)N
freq,1,1


Eighty-one molecules fail during the conversion, let's remove them from the dataset

In [24]:
print(smrt_dataset_df.shape)
smrt_dataset_df.dropna(axis=0, subset=['SMILES'], inplace=True)
print(smrt_dataset_df.shape)

(80019, 4)
(79938, 4)


## Save the Processed Dataset to File

In [26]:
smrt_dataset_df.to_csv(PROCESSED_SMRT_FILE, sep=";", index=False)